In [ ]:
import os
import sys
import pandas as pd
import xml.etree.ElementTree as et

def read_dataframe_from_csv(source_dir: str, file: str) -> pd.DataFrame:
    res = pd.read_csv(
        f"{source_dir}/{file}",     
        parse_dates=[4, 5],  # Parse columns 4 and 5 as dates
        infer_datetime_format=True,
        dayfirst=True)                
    
    # Convert both columns to strings before concatenation
    res['datetime'] = pd.to_datetime(res.iloc[:, 4].astype(str) + ' ' + res.iloc[:, 5].astype(str), dayfirst=True)
    res = res.drop(res.columns[[4, 5]], axis=1)  # Drop original date and time columns
    res.columns = ["latitude", "longitude", "sog", "cog", "datetime"]  # Reorder columns

    return res.set_index("datetime")

def dataframe_to_gpx_file(df: pd.DataFrame, filePath: str) -> None:    
    gpx = et.Element("gpx", attrib={"creator": "MS", "xmlns": "http://www.topografix.com/GPX/1/1"})  
    trk = et.SubElement(gpx, "trk")
    name = et.SubElement(trk, "name")
    name.text = "merged tracks"
    trkseg = et.SubElement(trk, "trkseg")

    def create_trkpt(lat: float, lon: float, time_index: pd.Timestamp) -> None:
        trkpt = et.SubElement(trkseg, "trkpt", attrib={"lat": str(lat), "lon": str(lon)})
        time = et.SubElement(trkpt, "time")
        time.text = str(time_index)

    for index, row in df.iterrows():                
        create_trkpt(row["latitude"], row["longitude"], index)           
    
    tree = et.ElementTree(gpx)
    et.indent(tree, space="\t", level=0)

    with open(filePath, "wb") as file:
        tree.write(file, encoding="utf-8")    

def parse(source_dir, res_file) -> None:         
    df = pd.concat(map(lambda file: read_dataframe_from_csv(source_dir, file), os.listdir(source_dir))).sort_index()
        
    dataframe_to_gpx_file(df, res_file)    

parse("/media/eko/NEW VOLUME", "./result.gpx")

/tmp/ipykernel_8321/1991085509.py:7: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  res = pd.read_csv(
/tmp/ipykernel_8321/1991085509.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  res = pd.read_csv(


TypeError: unsupported operand type(s) for +: 'DatetimeArray' and 'str'